In [38]:
from typing import Optional
from dotenv import load_dotenv
load_dotenv()
import openpyxl
import os
import pandas as pd
import logging as logger
from azure.core.credentials import AzureKeyCredential
from openai import AzureOpenAI, OpenAI
from app.utils.azure_ai_search_retriever import AzureAISearchRetriever, SearchCredential
from app.utils.rag_orchestrator import RAGOrchestrator
from typing import List
import base64
import openai
import weave
import re 

logger.getLogger().setLevel(logger.WARN)


In [2]:

def create_search_credential() -> SearchCredential:
    """Creates the appropriate Azure Search credential based on environment variables."""
    api_key = os.getenv("AZURE_SEARCH_API_KEY")
    if api_key:
        logger.info("Using Azure Search API Key credential.")
        return AzureKeyCredential(api_key)


def create_openai_client(purpose: str = "embedding") -> Optional[AzureOpenAI | OpenAI]:
    """Creates an AzureOpenAI client instance based on environment variables."""
    if purpose == "embedding":
        endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
        api_key = os.getenv("AZURE_OPENAI_API_KEY")
        api_version = os.getenv("AZURE_OPENAI_API_VERSION")
        log_prefix = "Embedding"
    elif purpose == "chat":
        endpoint = None
        api_version = None
        api_key = os.getenv("AZURE_OPENAI_CHAT_API_KEY", os.getenv("AZURE_OPENAI_API_KEY"))
        log_prefix = "Chat"
    else:
        raise ValueError(f"Invalid purpose for OpenAI client: {purpose}")

    try:
        if endpoint is None or api_version is None:
            client = OpenAI(
                api_key=api_key,
            )
        else:
            client = AzureOpenAI(
                azure_endpoint=endpoint,
                api_key=api_key,
                api_version=api_version,
            )
        # Perform a simple test if needed (e.g., list models), but might add latency
        logger.info(
            f"Azure OpenAI client for {log_prefix} initialized successfully (Endpoint: {endpoint}, API Version: {api_version}).")
        return client
    except Exception as e:
        logger.error(f"Failed to initialize Azure OpenAI {log_prefix} client: {e}", exc_info=True)
        return None  # Return None instead of raising, allow graceful failure if only one part is needed

def encode_file(file_path):
    with open(file_path, "rb") as file:
        return base64.b64encode(file.read()).decode('utf-8')


In [3]:
# --- Azure Search Configuration ---
search_service_name = os.getenv("AZURE_SEARCH_SERVICE_NAME")
index_name = os.getenv("AZURE_SEARCH_INDEX_NAME")
search_dns_suffix = os.getenv("AZURE_SEARCH_DNS_SUFFIX", "search.windows.net")

if not search_service_name or not index_name:
    raise ValueError("Missing required environment variables: AZURE_SEARCH_SERVICE_NAME and AZURE_SEARCH_INDEX_NAME")

search_service_endpoint = f"https://{search_service_name}.{search_dns_suffix}"
search_credential = create_search_credential()  # Handles API key, SPN, DefaultAzureCredential

# --- Azure OpenAI Configuration ---
# Embedding Client (Optional for Retriever if only doing text search)
openai_embedding_client = create_openai_client(purpose="embedding")
openai_embedding_deployment = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT")

# Chat Client (Required for Orchestrator)
openai_chat_client = create_openai_client(purpose="chat")
openai_chat_deployment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")

if not openai_chat_client or not openai_chat_deployment:
    raise ValueError(
        "Azure OpenAI Chat client configuration (Endpoint, Key, Version, Deployment) is required and incomplete.")

In [4]:
# --- Instantiate Retriever ---
retriever = AzureAISearchRetriever(
    search_service_endpoint=search_service_endpoint,
    index_name=index_name,
    search_credential=search_credential,
    openai_client=openai_embedding_client,  # Pass the client instance
    openai_embedding_deployment=openai_embedding_deployment  # Pass the deployment name
    # select_fields=None, # Use default
    # embedding_dimension=1536 # Use default
)
logger.info("AzureAISearchRetriever instantiated.")

# --- Instantiate Orchestrator ---
orchestrator = RAGOrchestrator(
    retriever=retriever,
    chat_client=openai_chat_client,  # Pass the client instance
    chat_deployment=openai_chat_deployment  # Pass the deployment name
    # system_prompt="Your custom system prompt here", # Optional
    # context_template="Source: {source}\nContent: {content}\n---\n", # Optional
)
logger.info("RAGOrchestrator instantiated.")

In [5]:
# --- Example Query ---
user_query = "Why do we need to do Business Process Re-engineering as a part of implementing an HIS/EHR? Note: Your answer must be in your own words."
logger.info(f"Answering query: '{user_query}'")

answer, sources = orchestrator.answer_query(
    user_query=user_query,
    top_k_retrieval=3,
    search_type="hybrid",  # Try hybrid search
    use_semantic_ranking=False  # Set to True if configured and desired
)

if answer:
    print("\n--- Answer ---")
    print(answer)
    print("\n--- Sources ---")
    if sources:
        for source in sources:
            print(f"- {source}")
    else:
        print("No sources were cited (or retrieval failed).")
else:
    print("\nFailed to get an answer.")

2025-04-01 03:28:23,759 - ERROR - Azure OpenAI API error during embedding generation: 404 - Error code: 404 - {'error': {'code': '404', 'message': 'Resource not found'}}
Traceback (most recent call last):
  File "c:\Users\MHS\Desktop\PROJECTS\ml-bu-autograder\app\utils\azure_ai_search_retriever.py", line 136, in _generate_embeddings
    response = self.openai_client.embeddings.create(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\MHS\Desktop\PROJECTS\ml-bu-autograder\venv\Lib\site-packages\openai\resources\embeddings.py", line 128, in create
    return self._post(
           ^^^^^^^^^^^
  File "c:\Users\MHS\Desktop\PROJECTS\ml-bu-autograder\venv\Lib\site-packages\openai\_base_client.py", line 1242, in post
    return cast(ResponseT, self.request(cast_to, opts, stream=stream, stream_cls=stream_cls))
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\MHS\Desktop\PROJECTS\ml-bu-autograder\venv\Lib\site-pack


--- Answer ---
I couldn't find any relevant documents to answer your question.

--- Sources ---
No sources were cited (or retrieval failed).


## Load Google Drive Files
Since our datasets live in Google Drive, we connect to our data source. This particular method assumes you have [Drive for Desktop](https://dl.google.com/drive-file-stream/GoogleDriveSetup.exe) installed on your computer and you are accessing a local path. For this POC, we focus on assignment 2.

In [6]:
class StudentResponse:
    rubric_path: str
    submission_path: str
    def __init__(self, rubric_path, submission_path):
        self.rubric_path = rubric_path
        self.submission_path = submission_path

In [7]:
import os

# Check if the directory exists and list its contents

base_dir = r"G:\My Drive\met_data\Quiz_1\24sprgmetcs581_o1 Quiz 1.xlsx"
if os.path.exists(base_dir):
    print("✅ Path exists!")
    print("Contents:", os.listdir(base_dir))  # List files/folders inside
else:
    print("❌ Path does NOT exist!")


✅ Path exists!


NotADirectoryError: [WinError 267] The directory name is invalid: 'G:\\My Drive\\met_data\\Quiz_1\\24sprgmetcs581_o1 Quiz 1.xlsx'

In [8]:
# Initialize list to hold student responses
student_responses: List[StudentResponse] = []
# Relevant material list
relevant_material = r"G:\My Drive\met_data\Assignment 2 – EHR Functional Requirements Worksheet\CS581 Assignment 2 HIS Clinical (EHR) Functional Requirements 2025 Spring 1.pdf"
assignment_requirements = r"G:\My Drive\met_data\Assignment 2 – EHR Functional Requirements Worksheet\CS581 Assignment 2 HIS Clinical (EHR) Functional Requirements 2025 Spring 1.pdf"

In [9]:
# Define the base directory
base_dir = r"G:\My Drive\met_data\Assignment 2 – EHR Functional Requirements Worksheet\24fallmetcs581_m1 submissions and rubrics"
# Iterate through student directories
for student_number in os.listdir(base_dir):
    student_path = os.path.join(base_dir, student_number)

    if os.path.isdir(student_path):  # Ensure it's a directory
        rubric_path = os.path.join(student_path, "rubric.docx")
        submission_path = os.path.join(student_path, "submission.pptx")

        # Check if both expected files exist
        if os.path.exists(rubric_path) and os.path.exists(submission_path):
            student_responses.append(StudentResponse(rubric_path=rubric_path, submission_path=submission_path))

In [10]:
response = openai_chat_client.responses.create(
    model="gpt-4o",
    input=[
        {
            "role": "system",
            "content": [
                {
                    "type": "input_text",
                    "text": "Please analyze the attached file."
                }
            ]
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "input_file",
                    "filename": os.path.basename(relevant_material),
                    "file_data": f"data:application/pdf;base64,{encode_file(relevant_material)}"
                }
            ]
        }
    ],
    text={
        "format": {
            "type": "text"
        }
    },
    reasoning={},
    tools=[],
    temperature=1,
    max_output_tokens=2048,
    top_p=1,
    store=True
)


In [11]:
from IPython.display import Markdown, display
display(Markdown(response.output[0].content[0].text))

### Overview of Assignment

**Course:** CS581 - Assignment 2  
**Title:** EHR Functional Requirements

**Objective:** Develop a high-level EHR Functional Requirements document for Virginia Women’s Center’s deployment of an EHR system, focusing on clinical functions. Align with CMS Medicare MIPS Quality Payment Program requirements.

### Requirements

1. **Functional Review:**
   - Examine functional requirements from the VWC case study.
   - Review CMS Promoting Interoperability requirements:
     - [MIPS Interoperability](https://qpp.cms.gov/mips/promoting-interoperability)
     - [2024 Program Requirements](https://www.cms.gov/medicare/regulations-guidance/promoting-interoperability-programs/calendar-year-2024-program-requirements)

2. **Use Worksheet:**
   - Use the provided Microsoft Excel Functional Requirements spreadsheet.
   - Identify top 10-12 functions needed for VWC and PI requirements.
   - Rate importance using a 1-4 scale.
   - Detail rationale in the Notes column.

3. **Submission:**
   - Submit via class Blackboard.
   - Follow the naming convention: `lastname_firstname Assignment 2`.
   - Include name in the worksheet.

### Grading Criteria

- **Understanding Objectives:** 
  - Grasp of EHR function and importance.

- **Key Functions Identification:**
  - Defined by lecture, textbook, and VWC case study.

- **Importance & Relevance:**
  - Explanation of function importance and relationship to VWC.
  - Consideration of government goals like MIPS, QPP, and PI.

- **Quality and Clarity:**
  - Material clarity, proper citations, and AI use documentation.

### Scoring

- **Understanding and Identification:** 45 Points
- **Importance and Relevance:** 45 Points
- **Clarity and Documentation:** 10 Points

**Total:** 100 Points

### Importance Scale

- **4:** Essential
- **3:** Important
- **2:** Nice
- **1:** Optional

**Vendor Score (not required):**

- **4:** Fully fulfills function
- **3:** Fulfills well
- **2:** Basic fulfillment
- **1:** Partial fulfillment
- **0:** Does not fulfill

This assignment focuses on integrating VWC needs with interoperability standards to improve healthcare service delivery through effective EHR functionality.


Test out with prompt technique to observe the varied results


In [12]:
response = openai_chat_client.responses.create(
    model="gpt-4o",
    input=[
        {
            "role": "system",
            "content": [
                {
                    "type": "input_text",
                    "text": """You are an expert document analyst. Analyze the attached file and provide a structured response with:  
                    1. A brief summary (5-7 sentences).  
                    2. The top 3 key insights.  
                    3. Any contradictions, biases, or missing information.  
                    4. Recommendations for improvement (if applicable).  
                    Format your response clearly with headings."""
                }
            ]
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "input_file",
                    "filename": os.path.basename(relevant_material),
                    "file_data": f"data:application/pdf;base64,{encode_file(relevant_material)}"
                }
            ]
        }
    ],
    text={"format": {"type": "text"}},
    reasoning={},
    tools=[],
    temperature=0.5,
    max_output_tokens=1500,
    top_p=0.9,
    store=True
)


In [13]:
from IPython.display import Markdown, display
display(Markdown(response.output[0].content[0].text))

## Summary

The document outlines an assignment for CS581, focusing on developing a high-level Electronic Health Record (EHR) Functional Requirements document for the Virginia Women’s Center (VWC). The assignment involves identifying essential clinical functions for an EHR system, guided by the VWC case study and CMS Medicare MIPS Quality Payment Program requirements. Students are tasked with using a Microsoft Excel spreadsheet to rate the importance of various functions on a scale of 1-4, explaining their choices in a notes column. The assignment emphasizes the integration of VWC’s needs with government goals like MIPS, QPP, and Promoting Interoperability. The grading rubric assesses understanding, identification of key functions, and clarity of material.

## Key Insights

1. **Integration with Government Programs**: The assignment highlights the need for EHR systems to align with government programs like MIPS and QPP, emphasizing interoperability.
   
2. **Importance Rating System**: A structured rating system is used to evaluate the criticality of each function, ensuring that essential functions are prioritized for VWC’s needs.

3. **Focus on Clinical Functions**: The assignment specifically targets the clinical aspects of EHR systems, requiring students to relate these functions to VWC’s operational requirements.

## Contradictions, Biases, or Missing Information

- **Lack of Detailed Case Study**: The document references a VWC case study, but without details, it's challenging to fully understand the specific needs and context.
- **Vendor Score Column Omission**: Students are instructed not to work on the vendor score column, potentially missing an opportunity to evaluate vendor capabilities.

## Recommendations for Improvement

1. **Provide Case Study Details**: Including a summary or key points from the VWC case study would help students better tailor their functional requirements.
   
2. **Clarify Vendor Evaluation**: Offer guidance on how vendor capabilities will be assessed in future assignments to provide a complete understanding of EHR system evaluation.

3. **Enhance Resource Links**: Ensure all links to external resources are current and accessible, aiding students in their research and understanding of government requirements.

In [14]:
excel = pd.read_excel(r'G:\My Drive\met_data\Quiz_1\24fallmetcs581_m1 Quiz 1.xlsx', sheet_name=None)
for sheet_name, df_sheet in excel.items():
    var_name = f"df_{sheet_name.replace(' ', '_')}"
    globals()[var_name] = df_sheet
    print(f"Assigned DataFrame to: {var_name}")


Assigned DataFrame to: df_Question_Details
Assigned DataFrame to: df_Student_Submissions


In [15]:
df_Student_Quiz_Responses = pd.read_excel(r'G:\My Drive\met_data\Quiz_1\24fallmetcs581_m1 Quiz 1.xlsx', sheet_name='Student Submissions')

In [16]:

pd.set_option('display.max_rows', 5)  # Limits rows displayed
pd.set_option('display.max_columns', 10)  # Limits columns displayed
df_Student_Quiz_Responses.columns

Index(['Unnamed: 0', 'question 13 answer', 'question 13 score',
       'question 13 feedback', 'question 14 answer', 'question 14 score',
       'question 14 feedback', 'additional feedback '],
      dtype='object')

In [17]:
def grade_student_response(course_material_path, student_response):
    # Encode the course material
    encoded_material = encode_file(course_material_path)
    
    # Create the grading prompt
    grading_prompt = f"""
    You are an expert grading assistant. Your task is to evaluate a student's response against the provided course material.
    
    Course Material Content (from attached file):
    [The system will have access to the attached course material]
    
    Student Response:
    {student_response}
    
   "As a health informatics professional, I will assess each student's response to the assignment based on the lecture material provided. The evaluation will include:

A score out of 14 reflecting the quality and completeness of the response.

Detailed feedback for each criterion, including an analysis of how well the response aligns with the concepts discussed in the lecture.

Specific suggestions for improvement to enhance the quality and depth of understanding.

Relevant quotes from the course material that support my evaluation and recommendations.

This grading process ensures that students receive constructive feedback tailored to the health informatics field, guiding them toward a deeper understanding of the subject matter.
    """
    
    response = openai_chat_client.responses.create(
        model="gpt-4o",
        input=[
            {
                "role": "system",
                "content": [
                    {
                        "type": "input_text",
                        "text": grading_prompt
                    }
                ]
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_file",
                        "filename": os.path.basename(course_material_path),
                        "file_data": f"data:application/pdf;base64,{encoded_material}"
                    }
                ]
            }
        ],
        text={"format": {"type": "text"}},
        reasoning={},
        tools=[],
        temperature=0.3,  # Lower temperature for more consistent grading
        max_output_tokens=2000,
        top_p=0.9,
        store=True
    )
    return (response.output[0].content[0].text)


In [18]:
COURSE_MATERIAL = r"G:\My Drive\met_data\Quiz_1\Mod 1 BPR & Workflow - Lecture Slides.pdf"
# Grade each answer in the DataFrame
df_Student_Quiz_Responses["question 13 grade"] = df_Student_Quiz_Responses["question 13 answer"].apply(
    lambda answer: grade_student_response(COURSE_MATERIAL, answer)
)


In [19]:
df_Student_Quiz_Responses

,Unnamed: 0,question 13 answer,question 13 score,question 13 feedback,question 14 answer,question 14 score,question 14 feedback,additional feedback,question 13 grade
0,student 1,since BPR shows and indicates the currents dat...,12 / 14,Your response covers several important points ...,Enterprise Architecture increase and enhance i...,14 / 14,Nice work!,Feedback on Short Answer Questions:\n\nQuestio...,**Evaluation of Student Response:**\n\n**Score...
1,student 2,When we look at the benefits from an IT operat...,14/14,Great work!,Business Process Re-engineering is very import...,14/14,Great work!,Question 13: Great work! \n\nQuestion 14: Grea...,**Evaluation of Student Response**\n\n**Score:...
...,...,...,...,...,...,...,...,...,...
20,student 21,\t\nOne significant benefit of using an Enterp...,13/14,"Overall, good work! Elaborating on how enhance...",Optimization of Workflows: BPR allows organiza...,14/14,Great job!,"Question 13: Overall, good work! Elaborating o...",**Score: 12/14**\n\n**Feedback:**\n\n1. **Unde...
21,student 22,Optimized use of resources and efforts.\n\nHav...,2025-11-14 00:00:00,You effectively identified a relevant benefit ...,\t\nI think that Business Process Re-engineeri...,2025-11-14 00:00:00,You’ve effectively highlighted the importance ...,Question 13: You effectively identified a rele...,**Score: 10/14**\n\n**Evaluation:**\n\n1. **Un...


In [20]:
COURSE_MATERIAL = r"G:\My Drive\met_data\Quiz_1\Mod 1 BPR & Workflow - Lecture Slides.pdf"

# Grade each answer in the DataFrame
df_Student_Quiz_Responses["question 14 grade"] = df_Student_Quiz_Responses["question 14 answer"].apply(
    lambda answer: grade_student_response(COURSE_MATERIAL, answer)
)


In [33]:
df_Student_Quiz_Responses

,Unnamed: 0,question 13 answer,question 13 score,question 13 feedback,question 14 answer,question 14 score,question 14 feedback,additional feedback,question 13 grade,question 14 grade
0,student 1,since BPR shows and indicates the currents dat...,12 / 14,Your response covers several important points ...,Enterprise Architecture increase and enhance i...,14 / 14,Nice work!,Feedback on Short Answer Questions:\n\nQuestio...,**Evaluation of Student Response:**\n\n**Score...,**Score: 12/14**\n\n**Feedback:**\n\n1. **Unde...
1,student 2,When we look at the benefits from an IT operat...,14/14,Great work!,Business Process Re-engineering is very import...,14/14,Great work!,Question 13: Great work! \n\nQuestion 14: Grea...,**Evaluation of Student Response**\n\n**Score:...,**Evaluation of Student Response**\n\n**Score:...
...,...,...,...,...,...,...,...,...,...,...
20,student 21,\t\nOne significant benefit of using an Enterp...,13/14,"Overall, good work! Elaborating on how enhance...",Optimization of Workflows: BPR allows organiza...,14/14,Great job!,"Question 13: Overall, good work! Elaborating o...",**Score: 12/14**\n\n**Feedback:**\n\n1. **Unde...,**Evaluation of Student Response**\n\n**Score:...
21,student 22,Optimized use of resources and efforts.\n\nHav...,2025-11-14 00:00:00,You effectively identified a relevant benefit ...,\t\nI think that Business Process Re-engineeri...,2025-11-14 00:00:00,You’ve effectively highlighted the importance ...,Question 13: You effectively identified a rele...,**Score: 10/14**\n\n**Evaluation:**\n\n1. **Un...,**Score: 10/14**\n\n**Evaluation:**\n\n1. **Un...


In [ ]:
def extract_score(text):
    score = re.search(r"Score: (\d{1,2}/14)", text)
    if score:
        return score.group(1)
    return None

# Apply the function to each row in the 'question grade' column
df_Student_Quiz_Responses['AI Q13 grade'] = df_Student_Quiz_Responses['question 13 grade'].apply(extract_score)
df_Student_Quiz_Responses['AI Q14 grade'] = df_Student_Quiz_Responses['question 14 grade'].apply(extract_score)



In [56]:
#preprocess the date to scores 
df_Student_Quiz_Responses['question 13 score'] = df_Student_Quiz_Responses['question 13 score'].astype(str)
df_Student_Quiz_Responses['question 14 score'] = df_Student_Quiz_Responses['question 14 score'].astype(str)
df_Student_Quiz_Responses['AI Q13 grade'] = df_Student_Quiz_Responses['AI Q13 grade'].astype(str)
df_Student_Quiz_Responses['AI Q14 grade'] = df_Student_Quiz_Responses['AI Q14 grade'].astype(str)

def convert_datetime_to_score(datetime_str):
    return f"{int(datetime_str[5:7])}/{int(datetime_str[8:10])}" if datetime_str.startswith("2025") else datetime_str

df_Student_Quiz_Responses['question 13 score'] = df_Student_Quiz_Responses['question 13 score'].str.replace(' ', '')

df_Student_Quiz_Responses['question 14 score'] = df_Student_Quiz_Responses['question 14 score'].str.replace(' ', '')
df_Student_Quiz_Responses['question 13 score'] = df_Student_Quiz_Responses['question 13 score'].apply(lambda x: convert_datetime_to_score(x).strip())
df_Student_Quiz_Responses['question 14 score'] = df_Student_Quiz_Responses['question 14 score'].apply(lambda x: convert_datetime_to_score(x).strip())
     

Exact Match Percentage

In [57]:
match_percentage = (df_Student_Quiz_Responses['question 13 score'] == df_Student_Quiz_Responses['AI Q13 grade']).mean() * 100

In [59]:
match_percentage = (df_Student_Quiz_Responses['question 14 score'] == df_Student_Quiz_Responses['AI Q14 grade']).mean() * 100

In [60]:
match_percentage


np.float64(4.545454545454546)